# 05 - Comparación de Modelos

Este notebook realiza una comparación exhaustiva de todos los modelos entrenados.

## Contenido
1. Carga de modelos entrenados
2. Comparación de métricas
3. Análisis estadístico
4. Curvas ROC y PR
5. Selección del mejor modelo

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data import SMSLoader, TextPreprocessor
from src.features import TextFeatureExtractor
from src.models import ClassicalModels
from src.evaluation import ModelEvaluator
from src.utils.visualization import Visualizer

%matplotlib inline

## 1. Carga de Datos y Modelos

In [ ]:
# Cargar datos
sms_loader = SMSLoader(data_dir='../data')

try:
    df = sms_loader.load_processed('sms_preprocessed.csv')
    text_col = 'text_clean_ml'
except:
    from src.data.sms_loader import create_sample_dataset
    df = create_sample_dataset()
    preprocessor = TextPreprocessor(lowercase=True, replace_urls=True)
    df['text_clean_ml'] = preprocessor.preprocess_batch(df['body'])
    text_col = 'text_clean_ml'

print(f"Dataset: {len(df)} mensajes")

In [ ]:
# Cargar o crear extractor de características
feature_extractor = TextFeatureExtractor(
    method='tfidf',
    max_features=5000,
    ngram_range=(1, 2)
)

X = feature_extractor.fit_transform(df[text_col])
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Entrenar modelos
models = ClassicalModels(model_dir='../models')

model_names = ['naive_bayes', 'logistic_regression', 'random_forest']

try:
    import xgboost
    model_names.append('xgboost')
except ImportError:
    pass

for name in model_names:
    print(f"Entrenando {name}...")
    models.train(name, X_train, y_train)

## 2. Evaluación Completa

In [ ]:
# Evaluador
evaluator = ModelEvaluator(output_dir='../reports/results')
viz = Visualizer(output_dir='../reports/figures')

# Evaluar todos los modelos
all_results = {}
roc_data = {}

for name in model_names:
    y_pred = models.predict(name, X_test)
    y_proba = models.predict_proba(name, X_test)
    
    metrics = evaluator.evaluate(y_test, y_pred, y_proba, model_name=name)
    all_results[name] = metrics
    
    # Guardar datos para curvas ROC
    fpr, tpr, _ = evaluator.get_roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, metrics.get('roc_auc', 0))

In [ ]:
# Tabla de comparación
comparison_df = evaluator.compare_models()
print("\n" + "="*60)
print("COMPARACIÓN DE MODELOS")
print("="*60)
display(comparison_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4))

## 3. Visualización de Resultados

In [ ]:
# Gráfico de barras comparativo
viz.plot_metrics_comparison(
    comparison_df,
    metrics=['accuracy', 'precision', 'recall', 'f1'],
    title='Comparación de Métricas por Modelo',
    save_path='model_metrics_comparison.png'
)

In [ ]:
# Curvas ROC comparativas
viz.plot_roc_curves_comparison(
    roc_data,
    title='Comparación de Curvas ROC',
    save_path='roc_curves_comparison.png'
)

## 4. Matrices de Confusión

In [ ]:
# Matrices de confusión para todos los modelos
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, name in zip(axes, model_names[:4]):
    cm = evaluator.results[name]['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legítimo', 'Smishing'],
                yticklabels=['Legítimo', 'Smishing'])
    ax.set_title(f'Matriz de Confusión - {name}')
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices_all.png', dpi=150)
plt.show()

## 5. Selección del Mejor Modelo

In [ ]:
# Mejor modelo por F1
best_f1_model, best_f1_score = evaluator.get_best_model(metric='f1')
print(f"Mejor modelo por F1: {best_f1_model} (F1: {best_f1_score:.4f})")

# Mejor modelo por ROC-AUC
best_auc_model, best_auc_score = evaluator.get_best_model(metric='roc_auc')
print(f"Mejor modelo por ROC-AUC: {best_auc_model} (AUC: {best_auc_score:.4f})")

In [ ]:
# Resumen del mejor modelo
print(f"\n{'='*60}")
print(f"RESUMEN DEL MEJOR MODELO: {best_f1_model}")
print('='*60)
evaluator.print_summary(best_f1_model)

## 6. Guardado de Resultados

In [ ]:
# Guardar comparación
evaluator.save_comparison('final_model_comparison.csv')

# Guardar mejor modelo
models.save_model(best_f1_model)

print("Resultados guardados exitosamente!")

## 7. Conclusiones

### Resultados principales:
- **Mejor modelo**: [modelo] con F1 de [score]
- **Mayor precision**: [modelo]
- **Mayor recall**: [modelo]

### Observaciones:
1. [Observación 1]
2. [Observación 2]
3. [Observación 3]

### Recomendación final:
Se recomienda usar [modelo] para producción debido a [razones].